<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 09
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        NSGA-II Portfolio Optimization
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Finds the best trade-off between expected return and risk for all the assets, using the expected returns from notebook 07 and the covariance matrix &Sigma; from notebook 08. A multi-objective genetic algorithm (NSGA-II) maps the whole return-risk front; the maximum-Sharpe portfolio on that front becomes the pipeline's allocation. Every run saves its tables (CSV) and charts to <code>../nsga2_optimization/&lt;run date&gt;/</code>; the takeaways explain how to read them.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Key Objectives in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Problem Definition:</b>Long-only weights, at most 10% per asset, RealEstate fixed at its share of the current portfolio (from <code>Consolidated_Portfolio_Positions.csv</code>), weights summing to 1, all on the 20-day horizon of the inputs.</li>
            <li><b>NSGA-II Search:</b> Two objectives &mdash; maximize expected return w<sup>T</sup>&mu;, minimize risk &radic;(w<sup>T</sup>&Sigma;w) &mdash; with a repair step that keeps every portfolio feasible.</li>
            <li><b>Exact Check:</b> The same problem solved exactly (efficient frontier), to measure how close NSGA-II gets.</li>
            <li><b>Portfolio Selection:</b> The maximum-Sharpe point of the front, with its holdings and class allocation, compared with the exact optimum and equal weight.</li>
            <li><b>Seed Stability:</b> Re-runs with other random seeds, so the allocation does not depend on luck.</li>
            <li><b>Save:</b> Weights and the full front for the benchmark comparison and backtest.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Benchmark Comparison &amp; Walk-Forward Backtest (five approaches)
    </p>
</div>

In [ ]:
#pip install pymoo
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
from src.config import DATA_DIR, PROCESSED_DIR
from src.helpers.charts import chart_dir
from src.optimization.positions import load_positions, portfolio_value, real_estate_weight, pnl_gap
from src.optimization.portfolio import build_spec, portfolio_stats, class_weights, equal_weight, check_feasible
from src.optimization.nsga2 import run_nsga2, select_max_sharpe
from src.optimization.exact_frontier import efficient_frontier, max_sharpe, min_volatility_at
from src.optimization.optimization_plots import (plot_pareto_front, plot_class_allocation_along_front,
                                                 plot_selected_allocation)

CHART_DIR = chart_dir(DATA_DIR, "nsga2_optimization")
MAX_WEIGHT = 0.10                                    # cap per asset
positions = load_positions(PROCESSED_DIR / "Consolidated_Portfolio_Positions.csv")
REAL_ESTATE_WEIGHT = real_estate_weight(positions)   # RealEstate value / total value of the current holdings
FIXED_CLASS_WEIGHTS = {"RealEstate": REAL_ESTATE_WEIGHT}
POP_SIZE, N_GEN, SEED = 200, 1000, 42                # NSGA-II settings
STABILITY_SEEDS = [7, 123, 2026]                     # extra seeds for Step 05
HORIZON = 20                                         # matches Expected_Return_20D and covariance_20d
ANNUAL_COLS = ["Return_Annual", "Volatility_Annual", "Sharpe_Annual"]

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Load Inputs &amp; Define the Portfolio Problem
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Loads the expected returns and volatilities from notebook 07 and &Sigma; from notebook 08, and fixes the rules every candidate portfolio must follow.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Constraints:</b> long-only (no short selling), at most <code>MAX_WEIGHT</code> per asset, RealEstate fixed at <code>REAL_ESTATE_WEIGHT</code> because it is an illiquid holding, weights summing to 1. With the defaults, 90% is spread over the remaining assets, so at least 9 holdings are needed.</li>
                        <li><b>RealEstate weight from current holdings:</b> <code>REAL_ESTATE_WEIGHT</code> = RealEstate value / total value in <code>Consolidated_Portfolio_Positions.csv</code> (Current Quantity &times; Current Price (EUR), rounded to 3 decimals; the RealEstate row has ISIN or Security Name <code>real estate</code>). It follows the real portfolio whenever the file is updated.</li>
            <li><b>PnL check:</b> the file's Unrealized PnL is compared with Current Quantity &times; (Current Price &minus; Average Buy Price). A gap near 0 confirms that the quantity and price columns are read as intended.</li>
            <li><b>Why long-only:</b> the return model showed no out-of-sample skill (k = 0 in notebook 07), so there is no basis for short positions, and shorting Bond or RealEstate is not meaningful. Long-only also limits the damage from estimation errors.</li>
            <li><b>Guards:</b> &Sigma; must be in the ticker order of the optimizer inputs and positive definite, and the cap must leave a feasible portfolio &mdash; any failure stops the run.</li>
            <li><b>Units:</b> the optimization runs on 20-day returns and risk; tables and charts show annualized values (return &times; 252/20, volatility &times; &radic;(252/20)), the same convention as notebook 07.</li>
        </ul>
    </div>
</div>

In [ ]:
opt = pd.read_parquet(DATA_DIR / "interim" / "optimizer_inputs.parquet")
sigma_df = pd.read_parquet(DATA_DIR / "interim" / "covariance_20d.parquet")
spec = build_spec(opt, sigma_df, max_weight=MAX_WEIGHT, fixed_class_weights=FIXED_CLASS_WEIGHTS)

In [ ]:
RF_ANNUAL = spec.rf * 252 / HORIZON
print(f"live date {pd.Timestamp(opt['Date'].iloc[0]):%Y-%m-%d}; results saved to {CHART_DIR}")
print(f"current portfolio {portfolio_value(positions):,.2f} EUR; RealEstate {REAL_ESTATE_WEIGHT:.1%} of it (fixed); "
      f"PnL check gap {pnl_gap(positions):.2f} EUR")

print(f"{len(spec.tickers)} assets: {len(spec.free_idx)} free, fixed {spec.fixed.round(4).to_dict()}")
print(f"budget for free assets {spec.budget:.0%}, cap {MAX_WEIGHT:.0%} -> at least "
      f"{int(np.ceil(spec.budget / MAX_WEIGHT - 1e-9))} holdings; risk-free rate {RF_ANNUAL:.2%} a year")

inputs = opt.assign(Return_Annual=opt["Expected_Return_20D"] * 252 / HORIZON,
                    Volatility_Annual=opt["Predicted_Volatility_Annual"])
inputs["Sharpe_Annual"] = (inputs["Return_Annual"] - RF_ANNUAL) / inputs["Volatility_Annual"]
input_summary = inputs.groupby("Asset_Class").agg(Assets=("Ticker", "size"),
                                                  Mean_Return_Annual=("Return_Annual", "mean"),
                                                  Max_Return_Annual=("Return_Annual", "max"),
                                                  Mean_Volatility_Annual=("Volatility_Annual", "mean"),
                                                  Mean_Asset_Sharpe=("Sharpe_Annual", "mean")).round(3)
display(input_summary)
input_summary.to_csv(CHART_DIR / "input_summary.csv")

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Input Summary
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Shows, per asset class, the expected returns and volatilities the optimizer works with. Saved as <code>input_summary.csv</code>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Columns:</b> number of assets, mean and highest expected return a year, mean volatility a year, and the mean Sharpe ratio of the single assets.</li>
            <li><b>What to expect:</b> each market class's equal-weighted portfolio has the same Sharpe ratio by construction (notebook 07, SR0), so classes with higher expected returns should also show higher volatility. Return differences reflect risk, not return forecasts.</li>
            <li><b>Single assets vs class portfolios:</b> a single asset's Sharpe ratio is normally below SR0, because each asset carries risk that cancels out inside its class portfolio. The lower <code>Mean_Asset_Sharpe</code> is in a class, the more the optimizer gains from spreading across that class.</li>
            <li><b>Bond and RealEstate:</b> Bond's expected return equals the risk-free rate, so its Sharpe ratio is 0 &mdash; it can lower risk but never adds excess return. RealEstate is pinned to its known return rather than the prior, so its Sharpe ratio need not match SR0; its weight is fixed in any case.</li>
            <li><b>What this means for the optimizer:</b> with equal class Sharpe ratios, the allocation is driven mainly by diversification &mdash; how the assets move together in &Sigma; &mdash; rather than by return views.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        NSGA-II: Searching the Return-Risk Trade-off
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Evolves a population of 200 portfolios for 1,000 generations. The result is the Pareto front: the portfolios for which no other portfolio has both a higher expected return and a lower risk.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Individual:</b> all the free weights (equals the number of assets); RealEstate is fixed, so it is not a gene. <b>Feasibility:</b> after crossover and mutation, every individual is replaced by the closest feasible portfolio (weights between 0 and the cap, summing to the budget), so no penalty terms are needed.</li>
            <li><b>Operators:</b> random starting portfolios of varying concentration, simulated binary crossover, polynomial mutation, non-dominated sorting and crowding distance (pymoo implementation).</li>
            <li><b>Objectives:</b> maximize expected return w<sup>T</sup>&mu; and minimize risk &radic;(w<sup>T</sup>&Sigma;w). Sharpe is read off the front afterwards. Sortino and turnover are measured in the backtest: the predicted inputs give no return distribution, and there is no previous portfolio at the live date.</li>
            <li><b>Reproducibility:</b> a fixed seed (<code>SEED = 42</code>); Step 05 repeats the run with other seeds.</li>
            <li>The return and risk range of the front is saved as <code>front_range.csv</code>.</li>
        </ul>
    </div>
</div>

In [ ]:
front = run_nsga2(spec, pop_size=POP_SIZE, n_gen=N_GEN, seed=SEED)
check_feasible(spec, front[spec.tickers].to_numpy())

print(f"{len(front)} Pareto-optimal portfolios, all feasible")
front_range = front[ANNUAL_COLS].agg(["min", "max"]).round(4)
display(front_range)
front_range.to_csv(CHART_DIR / "front_range.csv")

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Exact Check: NSGA-II vs the Efficient Frontier
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        With two objectives and linear constraints, the efficient frontier can also be computed exactly (a quadratic program). Solving it on the same inputs shows how close NSGA-II gets.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Risk gap:</b> each NSGA-II portfolio's risk compared with the lowest risk achievable at the same expected return. 0% means the portfolio lies on the exact frontier.</li>
            <li><b>Coverage:</b> whether the front reaches both ends &mdash; the minimum-variance and the maximum-return portfolio.</li>
            <li><b>Max-Sharpe comparison:</b> the best NSGA-II portfolio against the exact maximum-Sharpe portfolio; equal weight is shown for reference.</li>
            <li>The chart is saved to <code>../nsga2_optimization/&lt;run date&gt;/pareto_front.png</code>.</li>
        </ul>
    </div>
</div>

In [ ]:
exact = efficient_frontier(spec, n_points=50)
w_exact = max_sharpe(spec, exact)
risk_gap = front["Volatility_20D"].to_numpy() / min_volatility_at(spec, front["Return_20D"]) - 1

w_nsga = front.loc[select_max_sharpe(front), spec.tickers].to_numpy(dtype=float)
w_equal = equal_weight(spec)
check_feasible(spec, np.vstack([w_exact, w_equal]), tol=1e-7)

compare = portfolio_stats(spec, np.vstack([w_nsga, w_exact, w_equal]), horizon=HORIZON)
compare.index = ["NSGA-II max Sharpe", "Exact max Sharpe", "Equal weight"]
shortfall = 1 - compare.loc["NSGA-II max Sharpe", "Sharpe_20D"] / compare.loc["Exact max Sharpe", "Sharpe_20D"]

exact_check = pd.Series({
    "Median risk gap": f"{np.median(risk_gap):.2%}",
    "Largest risk gap": f"{risk_gap.max():.2%}",
    "Lowest return, NSGA-II vs exact": f"{front['Return_Annual'].min():.2%} vs {exact['Return_Annual'].min():.2%}",
    "Highest return, NSGA-II vs exact": f"{front['Return_Annual'].max():.2%} vs {exact['Return_Annual'].max():.2%}",
    "Max-Sharpe shortfall": f"{shortfall:.2%}",
}).to_frame("Value")
display(exact_check)
display(compare[ANNUAL_COLS].round(4))
exact_check.to_csv(CHART_DIR / "exact_check.csv")
compare[ANNUAL_COLS].to_csv(CHART_DIR / "portfolio_comparison.csv")
plot_pareto_front(front, exact, compare, save_dir=CHART_DIR)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Exact Check
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Measures how close NSGA-II gets to the exact optimum. Saved as <code>exact_check.csv</code> and <code>portfolio_comparison.csv</code>; chart <code>pareto_front.png</code>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Risk gap:</b> how much more risk an NSGA-II portfolio carries than the exact minimum at the same expected return. 0% means on the exact frontier; gaps of a few percent mean near-optimal. If they grow much larger, raise <code>N_GEN</code> or <code>POP_SIZE</code>.</li>
            <li><b>Coverage:</b> NSGA-II's lowest and highest expected return against the exact ends. Coverage matters most on the side of the max-Sharpe point: if the front stops short there, the selected portfolio is a cruder approximation.</li>
            <li><b>Max-Sharpe shortfall:</b> how far the best NSGA-II portfolio's predicted Sharpe ratio falls below the exact maximum &mdash; a measure of how well the search converged, not of how well the portfolio will perform. Which approach performs better is decided in the backtest.</li>
            <li><b>Why some gap is normal:</b> the exact optimum holds many weights at exactly 0 or exactly the cap, and a genetic algorithm searching n(number of asstes) continuous weights approaches such corners only slowly.</li>
            <li><b>In the chart:</b> the dots are the NSGA-II front, the line the exact frontier; equal weight sits below the frontier, because it ignores how the assets move together.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Selected Portfolio: Maximum Sharpe Ratio
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Selects the front portfolio with the highest predicted Sharpe ratio, (w<sup>T</sup>&mu; &minus; r<sub>f</sub>) / &radic;(w<sup>T</sup>&Sigma;w), and reports its holdings and class allocation.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Why max Sharpe:</b> it is the best expected return per unit of risk on the front, and it is the selection rule fixed before seeing any results.</li>
            <li><b>Class allocation:</b> asset weights summed per class, compared with the exact max-Sharpe portfolio and with equal weight.</li>
            <li><b>Bond:</b> it earns only the risk-free rate, yet holding it can raise the Sharpe ratio slightly: when other assets sit at their cap, Bond lets the rest of the portfolio carry less risk. Because that effect is small, NSGA-II gets little signal about where to put Bond, so its weight is often the least reliable number &mdash; check it in Step 05.</li>
            <li>Charts: the class mix along the whole front (<code>class_allocation_along_front.png</code>) and the selected allocation (<code>selected_class_allocation.png</code>).</li>
        </ul>
    </div>
</div>

In [ ]:
weights = pd.DataFrame({"Ticker": spec.tickers, "Asset_Class": spec.classes.loc[spec.tickers].to_numpy(),
                        "Weight": w_nsga, "Weight_Exact": w_exact})
held = weights[weights["Weight"] > 1e-4]
print(f"{len(held)} assets held (above 0.01%), {int((held['Weight'] >= MAX_WEIGHT - 1e-4).sum())} at the {MAX_WEIGHT:.0%} cap")
display(held.sort_values("Weight", ascending=False).round(4))

class_table = class_weights(spec, np.vstack([w_nsga, w_exact, w_equal])).T
class_table.columns = compare.index
display((class_table * 100).round(2))
held.sort_values("Weight", ascending=False).to_csv(CHART_DIR / "holdings.csv", index=False)
class_table.to_csv(CHART_DIR / "class_allocation.csv")

front_classes = class_weights(spec, front[spec.tickers].to_numpy())
plot_class_allocation_along_front(front_classes, front["Volatility_Annual"],
                                  compare.loc["NSGA-II max Sharpe", "Volatility_Annual"], save_dir=CHART_DIR)
plot_selected_allocation(class_table, save_dir=CHART_DIR)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Selected Portfolio
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Holdings and class allocation of the pipeline portfolio. Saved as <code>holdings.csv</code> and <code>class_allocation.csv</code>; charts <code>class_allocation_along_front.png</code> and <code>selected_class_allocation.png</code>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Holdings:</b> every asset above 0.01%, with the NSGA-II weight and the exact-optimum weight side by side. The printed line counts the holdings and the assets at the cap.</li>
            <li><b>Class allocation:</b> class weights (%) of the NSGA-II max-Sharpe portfolio, the exact max-Sharpe portfolio and equal weight. The gap between the first two shows, at class level, how far the search result is from the optimum.</li>
            <li><b>Along the front:</b> left is the lowest-risk end, right the highest-return end, and the dashed line marks the selected portfolio. Expect low-risk assets and diversifiers to dominate on the left and the highest-return classes on the right.</li>
            <li><b>Caveat:</b> all of these are predicted values. Which approach actually performs best is decided by the walk-forward backtest in notebook 10, not by this table.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 05
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Seed Stability
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Re-runs NSGA-II with three other seeds and compares the selected max-Sharpe portfolios. An allocation that depended on the seed would not be reliable.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><b>Class level:</b> the largest difference in any class weight across seeds.</li>
            <li><b>Asset level:</b> the largest difference in any single asset's weight. Asset weights can vary more than class weights, because assets within a class are close substitutes.</li>
            <li><b>Runtime:</b> one full NSGA-II run per extra seed.</li>
            <li>Saved as <code>seed_stability.csv</code> (class weights and Sharpe ratio per seed) and <code>seed_asset_weights.csv</code> (every asset's weight per seed, sorted by how much it varies).</li>
        </ul>
    </div>
</div>

In [ ]:
runs = {SEED: front}
for s in STABILITY_SEEDS:
    runs[s] = run_nsga2(spec, pop_size=POP_SIZE, n_gen=N_GEN, seed=s)

W_sel = np.vstack([f.loc[select_max_sharpe(f), spec.tickers].to_numpy(dtype=float) for f in runs.values()])
stability = class_weights(spec, W_sel)
stability.index = [f"seed {s}" for s in runs]
class_range = stability.max() - stability.min()
stability["Sharpe_Annual"] = portfolio_stats(spec, W_sel, horizon=HORIZON)["Sharpe_Annual"].to_numpy()
display(stability.round(4))
stability.to_csv(CHART_DIR / "seed_stability.csv")

print(f"largest class-weight difference across seeds: {100 * class_range.max():.2f} points ({class_range.idxmax()})")
asset_by_seed = pd.DataFrame(W_sel.T, index=spec.tickers, columns=stability.index)
asset_by_seed.insert(0, "Asset_Class", spec.classes.loc[spec.tickers].to_numpy())
asset_by_seed["Range"] = asset_by_seed[stability.index].max(axis=1) - asset_by_seed[stability.index].min(axis=1)
asset_by_seed = asset_by_seed.sort_values("Range", ascending=False)
asset_by_seed.to_csv(CHART_DIR / "seed_asset_weights.csv")
display(asset_by_seed.head(10).round(4))
print(f"largest asset-weight difference across seeds: {100 * asset_by_seed['Range'].iloc[0]:.2f} points ({asset_by_seed.index[0]})")

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Stability Check
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Compares the selected max-Sharpe portfolio across seeds. Saved as <code>seed_stability.csv</code> and <code>seed_asset_weights.csv</code>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Tables:</b> the first has one row per seed, with the class weights and the predicted Sharpe ratio a year. The second shows the 10 assets whose weight varies most across seeds; <code>Range</code> is the largest minus the smallest weight. The printed lines name the class and the asset with the largest difference.</li>
            <li><b>Class level:</b> differences of a few percentage points mean the class allocation is reliable. Larger differences mean the search has not settled: raise <code>N_GEN</code> or <code>POP_SIZE</code>, or report the class allocation as a range.</li>
            <li><b>Asset level:</b> differences are usually larger than at class level, because assets within a class are close substitutes. Read single-asset weights loosely.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 06
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Save the Pipeline Portfolio
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Saves the selected weights and the full Pareto front to <code>DATA_DIR/interim/</code>, next to the optimizer inputs.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><code>optimal_weights_pipeline.parquet</code>: Date, Ticker, Asset_Class and Weight of the selected portfolio, in the optimizer's ticker order.</li>
            <li><code>pareto_front_pipeline.parquet</code>: the weights and predicted statistics of every front portfolio.</li>
            <li>The weights file is reloaded and checked against the validated portfolio; notebook 10 compares this portfolio with the four benchmark approaches.</li>
        </ul>
    </div>
</div>

In [ ]:
out_dir = DATA_DIR / "interim"
weights_out = weights[["Ticker", "Asset_Class", "Weight"]].assign(Date=opt["Date"].iloc[0])
weights_out = weights_out[["Date", "Ticker", "Asset_Class", "Weight"]]
weights_out.to_parquet(out_dir / "optimal_weights_pipeline.parquet", index=False)
front.to_parquet(out_dir / "pareto_front_pipeline.parquet", index=False)

reloaded = pd.read_parquet(out_dir / "optimal_weights_pipeline.parquet")
assert reloaded["Ticker"].tolist() == spec.tickers, "saved order differs from optimizer_inputs"
assert np.array_equal(reloaded["Weight"].to_numpy(), w_nsga), "saved weights differ from the validated portfolio"
check_feasible(spec, reloaded["Weight"].to_numpy())
print(f"saved {len(reloaded)} weights and a {len(front)}-portfolio front to {out_dir}")